# Adaptive Boundary Sampling with Spike Interpolation

This notebook demonstrates the enhanced boundary condition sampling strategy that:
1. Detects spike events in boundary condition data
2. Creates interpolated sample points around spikes
3. Applies non-uniform sampling density with higher concentration near spikes
4. Visualizes the sampling distribution

In [ ]:
import sys
import os

# Add parent directory to path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.surf_flux import synth_surface_flux
from src.spike_detection import detect_spike_events
from src.adaptive_boundary_sampling import (
    adaptive_boundary_sampling,
    sample_boundary_points_with_interpolation,
    visualize_sampling_distribution
)
from src.boundary_sampling import sample_boundary_points

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

## Step 1: Generate Synthetic Data with Spikes

In [ ]:
# Generate synthetic surface flux data with spike events
t, q = synth_surface_flux(
    total_days=15,
    dt_minutes=30,
    storm_rate_per_day=0.4,
    min_storm_hours=0.5,
    max_storm_hours=0.5,
    seed=16
)

q_actual_flux = (t.tolist(), q.tolist())
q_tensor = torch.tensor(q_actual_flux[1], device=device)
t_tensor = torch.tensor(q_actual_flux[0], device=device).view(-1, 1)

print(f"Generated {len(t)} time points over {t[-1]/86400:.1f} days")
print(f"Flux range: [{q.min():.2e}, {q.max():.2e}] m/s")

## Step 2: Detect Spike Events

In [ ]:
# Detect spike events
spike_events, total_spike_indices = detect_spike_events(
    q_tensor,
    threshold_method='std',
    threshold_value=1.0,
    expansion_window=5,
    merge_distance=3,
    min_event_size=2,
    device=device
)

print(f"\nDetected {len(spike_events)} spike events")
for i, event in enumerate(spike_events):
    print(f"  Event {i+1}: {len(event)} points (indices {event.min().item()}-{event.max().item()})")

## Step 3: Visualize Original Spike Detection

In [ ]:
# Plot detected spikes
plt.figure(figsize=(14, 4))
plt.plot(t/86400, q, 'b-', alpha=0.5, label='Surface flux')

for i, event_indices in enumerate(spike_events):
    event_np = event_indices.cpu().numpy()
    plt.scatter(t[event_np]/86400, q[event_np], 
               label=f'Spike {i+1}', s=50, alpha=0.8)

plt.xlabel('Time (days)')
plt.ylabel('Surface Flux (m/s)')
plt.title('Detected Spike Events in Boundary Condition Data')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4: Compare Original vs Enhanced Sampling

### 4.1 Original Sampling (Event-Based Only)

In [ ]:
# Original sampling
batch_size_bc = 200
n_events = 9

t_bc_original = sample_boundary_points(
    t_tensor,
    spike_events,
    n_events,
    batch_size_bc,
    device
)

print(f"Original sampling: {len(t_bc_original)} samples")
print(f"Sample range: [{t_bc_original.min().item()/86400:.2f}, {t_bc_original.max().item()/86400:.2f}] days")

### 4.2 Enhanced Sampling (With Interpolation)

In [ ]:
# Enhanced adaptive sampling with interpolation
t_bc_enhanced = adaptive_boundary_sampling(
    t_tensor,
    spike_events,
    n_events,
    batch_size_bc,
    device=device,
    spike_ratio=0.7,              # 70% of samples from spike regions
    interpolation_density=3,      # 3 interpolated points between neighbors
    neighborhood_expansion=2,     # Expand spike regions by 2 points
    use_weighted_sampling=True    # Use importance sampling
)

print(f"\nEnhanced sampling: {len(t_bc_enhanced)} samples")
print(f"Sample range: [{t_bc_enhanced.min().item()/86400:.2f}, {t_bc_enhanced.max().item()/86400:.2f}] days")

### 4.3 Visualize Sampling Distributions

In [ ]:
# Visualize original sampling distribution
fig1 = visualize_sampling_distribution(
    t_tensor,
    q_tensor,
    spike_events,
    t_bc_original,
    title="Original Event-Based Sampling"
)
plt.show()

In [ ]:
# Visualize enhanced sampling distribution
fig2 = visualize_sampling_distribution(
    t_tensor,
    q_tensor,
    spike_events,
    t_bc_enhanced,
    title="Enhanced Adaptive Sampling with Interpolation"
)
plt.show()

## Step 5: Quantitative Comparison

In [ ]:
# Compute sampling statistics
def compute_spike_coverage(sampled_times, spike_events, t_tensor, tolerance=0.1):
    """
    Compute what fraction of samples fall within spike regions.
    
    Args:
        sampled_times: sampled time points
        spike_events: list of spike event indices
        t_tensor: all time points
        tolerance: time tolerance for matching (fraction of mean dt)
    """
    if len(spike_events) == 0:
        return 0.0
    
    # Get all spike time values
    all_spike_indices = torch.cat(spike_events).unique()
    spike_times = t_tensor[all_spike_indices].flatten()
    
    # Compute mean time step
    dt_mean = (t_tensor[1:] - t_tensor[:-1]).mean().item()
    tol = tolerance * dt_mean
    
    # Count samples within tolerance of spike times
    sampled_flat = sampled_times.detach().flatten()
    n_in_spike = 0
    
    for st in sampled_flat:
        min_dist = torch.abs(spike_times - st).min().item()
        if min_dist <= tol:
            n_in_spike += 1
    
    return n_in_spike / len(sampled_flat)

# Compare coverage
coverage_original = compute_spike_coverage(t_bc_original, spike_events, t_tensor, tolerance=2.0)
coverage_enhanced = compute_spike_coverage(t_bc_enhanced, spike_events, t_tensor, tolerance=2.0)

print("\nSampling Coverage Analysis:")
print("="*50)
print(f"Original sampling:")
print(f"  Spike region coverage: {coverage_original*100:.1f}%")
print(f"\nEnhanced sampling:")
print(f"  Spike region coverage: {coverage_enhanced*100:.1f}%")
print(f"\nImprovement: {(coverage_enhanced - coverage_original)*100:.1f} percentage points")
print("="*50)

## Step 6: Parameter Sensitivity Analysis

Test different interpolation densities and spike ratios

In [ ]:
# Test different parameter combinations
param_combinations = [
    {'spike_ratio': 0.5, 'interpolation_density': 2, 'neighborhood_expansion': 1},
    {'spike_ratio': 0.7, 'interpolation_density': 3, 'neighborhood_expansion': 2},
    {'spike_ratio': 0.8, 'interpolation_density': 5, 'neighborhood_expansion': 3},
]

print("\nParameter Sensitivity Analysis:")
print("="*70)

for i, params in enumerate(param_combinations):
    t_bc_test = adaptive_boundary_sampling(
        t_tensor,
        spike_events,
        n_events,
        batch_size_bc,
        device=device,
        **params
    )
    
    coverage = compute_spike_coverage(t_bc_test, spike_events, t_tensor, tolerance=2.0)
    
    print(f"\nConfig {i+1}: {params}")
    print(f"  Spike coverage: {coverage*100:.1f}%")

print("="*70)

## Step 7: Integration with PINN Training

Example of how to use the enhanced sampling in the training loop

In [ ]:
print("\nIntegration Example:")
print("="*70)
print("""
To use the enhanced sampling in your PINN training:

1. In train_loop.py, replace the line:
   
   from .boundary_sampling import sample_boundary_points
   
   with:
   
   from .adaptive_boundary_sampling import sample_boundary_points_with_interpolation

2. In the training loop, replace:
   
   t_bc = sample_boundary_points(q0_times_t, spike_events, n_events, batch_size_bc, device)
   
   with:
   
   t_bc = sample_boundary_points_with_interpolation(
       q0_times_t, spike_events, n_events, batch_size_bc, device,
       enable_interpolation=True,
       spike_ratio=0.7,
       interpolation_density=3,
       neighborhood_expansion=2
   )

3. The enhanced sampling will automatically:
   - Detect spike neighborhoods
   - Create interpolated sample points
   - Apply non-uniform sampling density
   - Focus 70% of samples on spike regions
""")
print("="*70)

## Summary

The enhanced adaptive boundary sampling provides:

1. **Spike Detection Enhancement**: Expands spike regions to include neighborhoods
2. **Interpolation**: Creates additional sample points between spike neighbors
3. **Non-Uniform Sampling**: Applies higher density near spike events
4. **Configurable**: Tunable parameters for different scenarios
5. **Backward Compatible**: Can fall back to original sampling if needed

Key parameters:
- `spike_ratio`: Fraction of batch from spike regions (0.5-0.8 recommended)
- `interpolation_density`: Number of interpolated points (2-5 recommended)
- `neighborhood_expansion`: Spike region expansion (1-3 recommended)
- `use_weighted_sampling`: Enable importance sampling (True recommended)